# 01 — Chunking Strategies

*Level 2 — Advanced RAG*

## Objective
Compare fixed-size, recursive, semantic, and parent-child chunking on a **real document** from the open BeIR/scifact corpus — the same corpus used throughout this level.


In [1]:
import sys
from pathlib import Path

LEVEL_DIR = Path.cwd().parent
sys.path.insert(0, str(LEVEL_DIR))
sys.path.insert(0, str(LEVEL_DIR / "hybrid-search"))
sys.path.insert(0, str(LEVEL_DIR / "query-transformations"))
sys.path.insert(0, str(LEVEL_DIR / "metadata-filtering"))
sys.path.insert(0, str(LEVEL_DIR / "context-compression"))


In [2]:
from common.dataset import prepare
from common.embed import OllamaEmbedder

data = prepare()
print(f"corpus={len(data.corpus)} queries={len(data.queries)}")

# Pick the longest document for a compelling chunking demo.
longest_id = max(data.doc_ids(), key=lambda d: len(data.corpus_text(d).split()))
text = data.corpus_text(longest_id)
print(f"Using doc {longest_id!r} — {len(text.split())} words")
print(text[:300], "...")


corpus=1000 queries=300
Using doc '27768226' — 1070 words
Open Access Increases Citation Rate PLoS Biology publishes today a research article by Gunther Eysenbach that is not about biology. It is about citations. It provides robust evidence that open-access articles (OA articles) are more immediately recognized and cited than non-OA articles. As such, it a ...


## Fixed-size (word count only, ignores structure)


In [3]:
from chunking.fixed_size import fixed_size_chunk

fixed_chunks = fixed_size_chunk(text, chunk_size=40, chunk_overlap=5)
print(f"{len(fixed_chunks)} chunks\n")
for i, c in enumerate(fixed_chunks[:3]):
    print(f"[{i}] {c}\n")


31 chunks

[0] Open Access Increases Citation Rate PLoS Biology publishes today a research article by Gunther Eysenbach that is not about biology. It is about citations. It provides robust evidence that open-access articles (OA articles) are more immediately recognized and cited than

[1] immediately recognized and cited than non-OA articles. As such, it adds objective support to the belief we have always held that open-access publication speeds up scientific dialog between researchers and, consequently, should be extended to the whole scientific literature as

[2] the whole scientific literature as quickly as possible. It is therefore fitting that we publish such a paper. We have long argued that papers freely available in a journal will be more often read and cited than those behind a



## Recursive (character-based, prefers sentence/paragraph boundaries)


In [4]:
from chunking.recursive import recursive_chunk

recursive_chunks = recursive_chunk(text, chunk_size=220, chunk_overlap=20)
print(f"{len(recursive_chunks)} chunks\n")
for i, c in enumerate(recursive_chunks[:3]):
    print(f"[{i}] {c}\n")


47 chunks

[0] Open Access Increases Citation Rate PLoS Biology publishes today a research article by Gunther Eysenbach that is not about biology. It is about citations

[1] It provides robust evidence that open-access articles (OA articles) are more immediately recognized and cited than non-OA articles

[2] As such, it adds objective support to the belief we have always held that open-access publication speeds up scientific dialog between researchers and, consequently, should be extended to the whole scientific literature



Notice recursive chunking's boundaries land at sentence ends (`. `) rather than mid-word — compare a fixed-size chunk boundary above against a recursive one here.


## Semantic (splits where meaning shifts, using real embeddings)


In [5]:
from chunking.semantic import semantic_chunk

embedder = OllamaEmbedder()
semantic_chunks = semantic_chunk(text, embed_fn=embedder.embed_one, breakpoint_percentile=75)
print(f"{len(semantic_chunks)} chunks\n")
for i, c in enumerate(semantic_chunks):
    print(f"[{i}] ({len(c.split())} words) {c[:200]}...\n")


11 chunks

[0] (24 words) Open Access Increases Citation Rate PLoS Biology publishes today a research article by Gunther Eysenbach that is not about biology. It is about citations....

[1] (88 words) It provides robust evidence that open-access articles (OA articles) are more immediately recognized and cited than non-OA articles. As such, it adds objective support to the belief we have always held...

[2] (16 words) However, solid evidence to support or refute such a claim has been surprisingly hard to find....

[3] (257 words) Since most open-access journals are new, comparisons of the effects of open access with established subscription-based journals are easily confounded by age and reputation. In the current study, Eysen...

[4] (20 words) Yes, you're right; we do have a strong and vested interest in publishing results that so obviously endorse our existence....

[5] (14 words) Moreover, the author of the article is also an editor of an open-access journal....

[6] (66 words) But s

## Parent-child (small chunks for retrieval, large chunks for context)


In [6]:
from chunking.parent_child import chunk_with_parents

children = chunk_with_parents(text, parent_size=60, child_size=15, child_overlap=3)
print(f"{len(children)} child chunks across {len({c.parent_id for c in children})} parents\n")
for c in children[:3]:
    print(f"child (parent={c.parent_id}): {c.text}")
print(f"\n...parent {children[0].parent_id} in full: {children[0].parent_text[:200]}...")


89 child chunks across 18 parents

child (parent=0): Open Access Increases Citation Rate PLoS Biology publishes today a research article by Gunther Eysenbach
child (parent=0): by Gunther Eysenbach that is not about biology. It is about citations. It provides robust
child (parent=0): It provides robust evidence that open-access articles (OA articles) are more immediately recognized and cited

...parent 0 in full: Open Access Increases Citation Rate PLoS Biology publishes today a research article by Gunther Eysenbach that is not about biology. It is about citations. It provides robust evidence that open-access ...


## What I observed

- **Fixed-size** cuts mid-sentence whenever the word count runs out — fast and simple, but breaks facts arbitrarily.
- **Recursive** respects sentence boundaries whenever possible, falling back to finer separators only when a piece is still too large.
- **Semantic** chunk boundaries follow topic shifts, not length — chunk count and size vary with content, not a fixed rule.
- **Parent-child** indexes small, precise child chunks for retrieval while keeping the larger parent available for generation context.

## Next

[02 — Dense vs. Sparse Retrieval](./02_dense_vs_sparse_retrieval.ipynb)
